# 00 · Colab bootstrap — Business Entity Resolution
Run this at the start of **every** Colab session (Runtime: **L4 GPU + High-RAM**).
It mounts Drive, copies the dataset to fast local disk, fetches the code, installs
dependencies and runs the requested stage. Only the provided challenge files are used;
nothing is looked up online.

**Before the first run:** edit the settings cell below.

In [ ]:
# ==== SETTINGS — edit once ====
DRIVE_ROOT  = "/content/drive/MyDrive/amazon_ml_2026"          # project folder in your Drive
DATA_ZIP    = f"{DRIVE_ROOT}/dataset.zip"                      # zip of the challenge 'dataset' folder
GITHUB_REPO = "https://github.com/<your-username>/business-entity-resolution.git"
CODE_ZIP    = f"{DRIVE_ROOT}/code.zip"                         # fallback if you don't use GitHub
WORK_DIR    = f"{DRIVE_ROOT}/work"                             # checkpoints/reports survive disconnects

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Copy + unzip the dataset to local disk (reading from Drive directly is slow)
import os, glob, shutil, subprocess, time
LOCAL_DATA = "/content/data"
if not glob.glob(f"{LOCAL_DATA}/**/train/train_source1.tsv", recursive=True):
    t = time.time(); os.makedirs(LOCAL_DATA, exist_ok=True)
    shutil.copy(DATA_ZIP, "/content/dataset.zip")
    subprocess.run(["unzip", "-q", "-o", "/content/dataset.zip", "-d", LOCAL_DATA], check=True)
    os.remove("/content/dataset.zip")
    print(f"dataset unzipped in {time.time() - t:.0f}s")
hit = glob.glob(f"{LOCAL_DATA}/**/train/train_source1.tsv", recursive=True)
assert hit, "train/train_source1.tsv not found - check how the zip was made"
DATA_DIR = os.path.dirname(os.path.dirname(hit[0]))
os.environ["BER_DATA_DIR"] = DATA_DIR
os.makedirs(WORK_DIR, exist_ok=True); os.environ["BER_WORK_DIR"] = WORK_DIR
print("BER_DATA_DIR =", DATA_DIR)
for split in ("train", "test"):
    for f in sorted(os.listdir(f"{DATA_DIR}/{split}")):
        print(f"  {split}/{f}: {os.path.getsize(f'{DATA_DIR}/{split}/{f}') / 1e6:,.0f} MB")

In [ ]:
# Fetch the code: private GitHub repo (token in Colab Secrets as GITHUB_TOKEN) or code.zip on Drive
CODE_DIR = "/content/code"
if os.path.isdir(f"{CODE_DIR}/.git"):
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q", "--ff-only"], check=True)
    print("code updated (git pull)")
else:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    if token and "<your-username>" not in GITHUB_REPO:
        url = GITHUB_REPO.replace("https://", f"https://{token}@")
        r = subprocess.run(["git", "clone", "-q", url, CODE_DIR], capture_output=True)
        if r.returncode != 0:
            raise RuntimeError("git clone failed - check the repo URL and token permissions")
        print("code cloned from GitHub")
    else:
        shutil.rmtree("/content/code_unzip", ignore_errors=True)
        subprocess.run(["unzip", "-q", "-o", CODE_ZIP, "-d", "/content/code_unzip"], check=True)
        CODE_DIR = os.path.dirname(os.path.dirname(
            glob.glob("/content/code_unzip/**/src/eda.py", recursive=True)[0]))
        print("code unzipped from Drive")
print("CODE_DIR =", CODE_DIR)
!git -C {CODE_DIR} log --oneline -3 2>/dev/null || true

In [ ]:
!pip install -q -r {CODE_DIR}/requirements.txt
import polars as pl, psutil
print("polars", pl.__version__, "| cores", os.cpu_count(),
      "| RAM", round(psutil.virtual_memory().total / 1e9, 1), "GB")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true

## Stage: EDA (full data, roughly 5–15 min)

In [ ]:
!cd {CODE_DIR} && python -m src.eda --out-dir {WORK_DIR}/eda

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK_DIR}/eda/eda_report.md", encoding="utf-8").read()))

## Stage: normalize (all six source files -> parquet, a few minutes)

In [ ]:
!cd {CODE_DIR} && python -m tests.test_normalize && python -m src.normalize

### When finished
Download `eda_report.md` and `match_examples.tsv` from `WORK_DIR/eda` (they are also in your Drive),
then **Runtime → Disconnect and delete runtime** so compute units stop being used.